In [0]:
dbutils.widgets.text("query_name", "SLD_FCST", "Query Name")
dbutils.widgets.text("run_date", "2026-01-21", "Run Date")
dbutils.widgets.text("market_run_id", "DAM", "Market Run ID")

In [0]:
from datetime import datetime, timedelta

def get_dam_window(run_date: str):
    """
    run_date: yyyy-MM-dd
    returns startdatetime, enddatetime for CAISO DAM
    """
    run_dt = datetime.strptime(run_date, "%Y-%m-%d")

    start_dt = (run_dt + timedelta(days=1)).strftime("%Y%m%d") + "T08:00-0000"
    end_dt   = (run_dt + timedelta(days=2)).strftime("%Y%m%d") + "T08:00-0000"

    return start_dt, end_dt


In [0]:
run_date = dbutils.widgets.get("run_date")

print(run_date)

In [0]:
run_date = dbutils.widgets.get("run_date")

start_dt, end_dt = get_dam_window(run_date)


In [0]:
market_run_id = dbutils.widgets.get("market_run_id")


In [0]:
query_name = dbutils.widgets.get("query_name")

url = f"https://oasis.caiso.com/oasisapi/SingleZip?&resultformat=6&queryname={query_name}&version=1&market_run_id={market_run_id}&startdatetime={start_dt}&enddatetime={end_dt}"

print(url)

In [0]:
import requests

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"API failed: {response.status_code}")

zip_path = f"/Volumes/workspace/bronze/caiso_api/api_zip_files/{query_name}.zip"

with open(zip_path, "wb") as f:
    f.write(response.content)


In [0]:
dbutils.fs.ls('/Volumes/workspace/bronze/caiso_api/api_zip_files/')

In [0]:
import zipfile
import os

extract_path = f"/Volumes/workspace/bronze/caiso_api/api_unzip_files/{query_name}/{market_run_id}"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)


In [0]:
dbutils.fs.ls('/Volumes/workspace/bronze/caiso_api/api_unzip_files/SLD_FCST/DAM/')

In [0]:
df = spark.read.csv('/Volumes/workspace/bronze/caiso_api/api_unzip_files/SLD_FCST/DAM/20260121_20260122_SLD_FCST_DAM_20260121_04_42_29_v1.csv', header=True)

df.display()

In [0]:
df.display()